# Module 05 — Exercise: Optical Image Classification

**DIGIHAZ PhD Course** | Disaster Risk Reduction


## Student Information

- **Name:** Yue Qimin
- **Student ID:** Not provided
- **Date:** 2026-10-01


## 1. Sentinel / Planet and optical disaster mapping

High-resolution and very-high-resolution optical imagery is useful for pre/post-event comparison because spectral and spatial changes can be interpreted directly. Sentinel-2 provides multispectral coverage suitable for broad-area mapping, while higher-resolution commercial constellations can provide finer spatial detail and higher revisit frequency.

Manual photo-interpretation is context-rich but subjective and difficult to scale. Automated pixel-based methods are faster but can ignore object shape and spatial context. Object-Based Image Analysis (OBIA) addresses this by grouping neighboring pixels into meaningful objects before classification.


## 2. OBIA vs. pixel-based classification

OBIA can use **shape, texture, neighborhood and spatial context**, not just the spectral value of one pixel. This is especially useful in damage mapping because buildings, roads and debris fields are spatial objects. Pixel-based methods remain useful for rapid large-area screening, but may produce salt-and-pepper noise and confusion between spectrally similar classes.


## 3. Deep Learning and classical Machine Learning

Classical machine-learning workflows usually depend on manually selected features and a separate classifier (e.g., k-means, random forest, SVM). Deep learning learns hierarchical representations directly from data and can combine diverse cues, but it normally needs more computation, careful training data and validation.

The lecture exercise describes a pipeline using a **Panopticon remote-sensing foundation model**, followed by **PCA** for low-dimensional embeddings and **k-means** for segmentation.


## 4. Executable PCA + k-means demonstration

To keep this notebook self-contained, the following reproduces the analytical stages using a synthetic feature cube. The same PCA/k-means logic can be applied to model embeddings.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

rng = np.random.default_rng(7)
h, w, bands = 96, 96, 8

# Three synthetic land-cover / damage regimes
labels_true = np.zeros((h,w), dtype=int)
labels_true[:, 32:64] = 1
labels_true[:, 64:] = 2

means = np.array([
    [0.2,0.25,0.3,0.55,0.62,0.5,0.35,0.3],
    [0.45,0.42,0.38,0.32,0.3,0.28,0.4,0.44],
    [0.12,0.14,0.18,0.22,0.25,0.2,0.16,0.13]
])

cube = np.zeros((h,w,bands))
for k in range(3):
    mask = labels_true == k
    cube[mask] = rng.normal(means[k], 0.035, (mask.sum(), bands))

X = cube.reshape(-1, bands)
pca = PCA(n_components=3, random_state=0)
Z = pca.fit_transform(X)

km = KMeans(n_clusters=3, random_state=0, n_init=20)
clusters = km.fit_predict(Z).reshape(h,w)

print("Explained variance by first 3 PCs:", np.round(pca.explained_variance_ratio_,3))
print("Total explained variance:", round(pca.explained_variance_ratio_.sum(),3))


In [ ]:
fig, axes = plt.subplots(1,3,figsize=(12,4))
axes[0].imshow(labels_true, cmap="tab10"); axes[0].set_title("Synthetic reference")
axes[1].imshow(Z[:,0].reshape(h,w), cmap="viridis"); axes[1].set_title("PC1 embedding")
axes[2].imshow(clusters, cmap="tab10"); axes[2].set_title("k-means segmentation")
for ax in axes: ax.axis("off")
plt.tight_layout()
plt.show()


## 5. Pipeline hyperparameters

Important hyperparameters to test include:
- model architecture / checkpoint;
- input patch size and spatial resolution;
- PCA component count;
- k-means cluster count `k`;
- k-means initialization, repetitions (`n_init`) and random seed;
- normalization / standardization before PCA;
- any spatial smoothing or object-merging rules after clustering.

A useful experiment changes one parameter at a time and records cluster stability, interpretability and agreement with reference data.


## 6. Discussion answers

**Why is OBIA more powerful than a pixel-based approach?**  
Because disaster objects have geometry and context. OBIA can use object shape, texture and neighborhood relationships, reducing isolated pixel noise and enabling more semantically meaningful classes.

**Is Deep Learning useful for modern remote sensing?**  
Yes, especially when multiple sensors, scales and contextual cues need to be integrated. Its advantage is learned feature representation, but its outputs still require domain-aware validation and uncertainty reporting.
